In [27]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
# @title Libraries
from typing import Mapping
import os
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms
import torchvision.models as models
from PIL import Image
import numpy as np
import librosa
import matplotlib.pyplot as plt
from torch import optim
import time
import random
import matplotlib.axes as ax
import pandas as pd
import seaborn as sn

In [29]:
# @title Constants
class_to_int = {
    'neutral' : 0,
    'calm' : 1,
    'happy' : 2,
    'sad': 3,
    'angry': 4,
    'fearful': 5,
    'disgust': 6,
    'surprised': 7
}

int_to_class = {v: k for k, v in class_to_int.items()}

data_pth = '/content/drive/MyDrive/UC_COSMOS/RAVDESS_DATASET/combined'
arr_pth = '/content/drive/MyDrive/UC_COSMOS/EmoNet/data_arrs'

SAMPLE_RATE = 48000

train_arrs = os.path.join(arr_pth, 'train_arrs')
test_arrs = os.path.join(arr_pth, 'test_arrs')

In [30]:
# @title Custom Dataset
def add_noise(signal, num_bits=16, augmented_num=2, snr=30):
    signal_len = len(signal)
    # Generate White Gaussian noise
    noise = np.random.normal(size=(augmented_num, signal_len))
    # Normalize signal and noise
    norm_constant = 2.0**(num_bits-1)
    signal_norm = signal / norm_constant
    noise_norm = noise / norm_constant
    # Compute signal and noise power
    s_power = np.sum(signal_norm ** 2) / signal_len
    n_power = np.sum(noise_norm ** 2, axis=1) / signal_len
    # Random SNR: Uniform [15, 30] in dB
    target_snr = os.ST_NODIRATIME
    # Compute K (covariance matrix) for each noise
    K = np.sqrt((s_power / n_power) * 10 ** (- target_snr / 10))
    K = np.ones((signal_len, augmented_num)) * K
    # Generate noisy signal
    return signal + K.T * noise

def add_noise_image(image, mean=0, var=0.01):
    image_arr = np.array(image)
    sigma = np.sqrt(var)

    # Generate Gaussian noise
    gauss_noise = np.random.normal(mean, sigma, image_arr.shape)

    # Add noise to the image
    noisy_image_array = image_arr + gauss_noise

    # Clip the values to be in valid range
    noisy_image_array = np.clip(noisy_image_array, 0, 255)

    noisy_image = Image.fromarray(np.uint8(noisy_image_array))

    return noisy_image

# Define the transforms for grayscale images
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

class CombinedDataset(Dataset):
    def __init__(self, root_dir, transform=transform, max_frames=35, offset=0, add_noise=True, noise_idx=0, sound_snr=30, image_var=0):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = os.listdir(root_dir)
        self.class_to_idx = class_to_int
        self.video_files = []
        self.audio_files = []
        self.labels = []
        self.max_frames = max_frames
        self.offset = offset
        self.add_noise = add_noise
        self.noise_idx = noise_idx
        self.sound_snr = sound_snr
        self.image_var = image_var

        for class_name in self.classes:
            class_dir = os.path.join(self.root_dir, class_name)
            if os.path.isdir(class_dir):
                for mini_dir in os.listdir(class_dir):
                    mini_path = os.path.join(class_dir, mini_dir)
                    files = os.listdir(mini_path)
                    if files[0].endswith('.mp4'):
                        self.video_files.append(os.path.join(mini_path, files[0]))
                        self.audio_files.append(os.path.join(mini_path, files[1]))
                        self.labels.append(self.class_to_idx[class_name])
                    else:
                        self.video_files.append(os.path.join(mini_path, files[1]))
                        self.audio_files.append(os.path.join(mini_path, files[0]))
                        self.labels.append(self.class_to_idx[class_name])

    def __len__(self):
        return len(self.video_files)

    def __getitem__(self, idx):
        #print(f"Worker processing index {idx}")
        video_path = self.video_files[idx]
        audio_path = self.audio_files[idx]
        label = self.labels[idx]
        #print(f"Worker {idx} beginning frames")
        frames = self.get_vid(video_path)
        #print(f"Worker {idx} getting audio")
        spec = self.get_spec(audio_path)
        #print(f"Worker {idx} finished!")
        return frames, spec, label

    def print_files(self):
        for i in range(len(self.video_files)):
            print('Video file: ' + self.video_files[i].split('/')[-1] + ' Audio file: ' + self.audio_files[i].split('/')[-1] + ' label: ' + str(self.labels[i]))

    def get_spec(self, path):
        t = time.time()
        audio, sample_rate = librosa.load(path, duration=3, sr=SAMPLE_RATE)
        signal = np.zeros((int(SAMPLE_RATE*3, )))
        signal[:len(audio)] = audio
        if (self.add_noise):
            signal = add_noise(signal)[self.noise_idx]
        mel_spec = librosa.feature.melspectrogram(y=signal, sr=sample_rate, n_fft=1024, win_length = 512, window='hamming', hop_length = 256, n_mels=128, fmax=sample_rate/2)
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
        """
        print(mel_spec_db.shape)
        plt.figure(figsize=(12, 4))
        librosa.display.specshow(mel_spec_db, sr=SAMPLE_RATE, x_axis='time', y_axis='hz')
        plt.colorbar(format='%+2.0f dB')
        plt.title('Spectrogram')
        plt.show()
        """
        #print("Time Audio: " + str(time.time() - t))
        return torch.tensor(mel_spec_db)


    def get_vid(self, video_path):
        cap = cv2.VideoCapture(video_path)
        frames = []
        t = time.time()
        for i in range(self.max_frames):
            ret, frame = cap.read()
            if not ret:
                break

            # Convert frame to grayscale
            RGB_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            faces = face_cascade.detectMultiScale(RGB_frame, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
            if len(faces) > 0:
                # Sort faces by size (largest face first)
                faces = sorted(faces, key=lambda x: x[2] * x[3], reverse=True)

                # Take the first detected face (largest face)
                (x, y, w, h) = faces[0]

                # Crop the face region from the frame
                face = frame[y:y+h, x:x+w]

                # Convert to PIL Image for transformation
                pil_face = Image.fromarray(face)

                if self.transform:
                    face_tensor = self.transform(pil_face)
                    face_tensor = torch.squeeze(face_tensor)  # Remove single-dimensional entries
                    frames.append(face_tensor)
        cap.release()

        frames = torch.stack(frames)
        frames = frames.permute(1, 0, 2, 3)
        return frames

# @title Datasets (Not Used ATM)
train_dataset = ConcatDataset([CombinedDataset(os.path.join(data_pth, 'train'),add_noise=False),
                               CombinedDataset(os.path.join(data_pth, 'train'),add_noise=False),
                               CombinedDataset(os.path.join(data_pth, 'train'),add_noise=False)])
test_dataset = ConcatDataset([CombinedDataset(os.path.join(data_pth, 'test'),add_noise=False),
                              CombinedDataset(os.path.join(data_pth, 'test'),add_noise=False),
                              CombinedDataset(os.path.join(data_pth, 'test'),add_noise=False)])


In [31]:
train_arr = []
ls = [[], [], []]
for i in range(len(train_dataset)):
    train = train_dataset[i]
    ls[0].append(train[0])
    ls[1].append(train[1])
    ls[2].append(train[2])
    print('counter: ' + str(i))
    if (len(ls[0]) == 32):
        train_arr.append(ls)
        ls = [[], [], []]

Streaming output truncated to the last 5000 lines.
counter: 403
counter: 404
counter: 405
counter: 406
counter: 407
counter: 408
counter: 409
counter: 410
counter: 411
counter: 412
counter: 413
counter: 414
counter: 415
counter: 416
counter: 417
counter: 418
counter: 419
counter: 420
counter: 421
counter: 422
counter: 423
counter: 424
counter: 425
counter: 426
counter: 427
counter: 428
counter: 429
counter: 430
counter: 431
counter: 432
counter: 433
counter: 434
counter: 435
counter: 436
counter: 437
counter: 438
counter: 439
counter: 440
counter: 441
counter: 442
counter: 443
counter: 444
counter: 445
counter: 446
counter: 447
counter: 448
counter: 449
counter: 450
counter: 451
counter: 452
counter: 453
counter: 454
counter: 455
counter: 456
counter: 457
counter: 458
counter: 459
counter: 460
counter: 461
counter: 462
counter: 463
counter: 464
counter: 465
counter: 466
counter: 467
counter: 468
counter: 469
counter: 470
counter: 471
counter: 472
counter: 473
counter: 474
counter: 475


In [32]:
ls = [[], [], []]
shp = train_arr[0][0][0].shape
for i in range(len(train_arr)):
    for j in range(len(train_arr[i][0])):
        print(train_arr[i][0][j].shape)
        if (train_arr[i][0][j].shape == shp):
            for k in range(len(train_arr[i])):
                ls[k].append(train_arr[i][k][j])

Streaming output truncated to the last 5000 lines.
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 224, 224])
torch.Size([3, 35, 

In [33]:
print(len(ls[0]))

5370


In [34]:
import os
import torch
import random


arr_pth = '/content/drive/MyDrive/UC_COSMOS/EmoNet/data_arrs'
train_arrs = os.path.join(arr_pth, 'train_arrs')
train_batches = os.path.join(train_arrs, '')
class_to_int = {
    'neutral' : 0,
    'calm' : 1,
    'happy' : 2,
    'sad': 3,
    'angry': 4,
    'fearful': 5,
    'disgust': 6,
    'surprised': 7
}

int_to_class = {v: k for k, v in class_to_int.items()}

In [35]:

global_cntr = {
    'neutral':0,
    'calm':0,
    'happy':0,
    'sad':0,
    'angry':0,
    'fearful':0,
    'disgust':0,
    'surprised':0
}
for i, batch in enumerate(os.listdir(train_batches)):
    cntr = {
        'neutral':0,
        'calm':0,
        'happy':0,
        'sad':0,
        'angry':0,
        'fearful':0,
        'disgust':0,
        'surprised':0
    }
    labels = torch.load(os.path.join(train_batches, batch))[2]
    for label in labels:
        cntr[int_to_class[label.item()]] += 1
        global_cntr[int_to_class[label.item()]] += 1
    print(f'Batch {i+1}: {cntr}')

<ipython-input-35-5c2e91b77f17>:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  labels = torch.load(os.path.join(train_batches, batch))[2]


Batch 1: {'neutral': 3, 'calm': 2, 'happy': 7, 'sad': 6, 'angry': 6, 'fearful': 3, 'disgust': 4, 'surprised': 1}
Batch 2: {'neutral': 4, 'calm': 5, 'happy': 4, 'sad': 4, 'angry': 8, 'fearful': 3, 'disgust': 3, 'surprised': 1}
Batch 3: {'neutral': 2, 'calm': 10, 'happy': 1, 'sad': 6, 'angry': 3, 'fearful': 5, 'disgust': 3, 'surprised': 2}
Batch 4: {'neutral': 0, 'calm': 4, 'happy': 3, 'sad': 8, 'angry': 8, 'fearful': 3, 'disgust': 2, 'surprised': 4}
Batch 5: {'neutral': 2, 'calm': 5, 'happy': 2, 'sad': 2, 'angry': 6, 'fearful': 4, 'disgust': 3, 'surprised': 8}
Batch 6: {'neutral': 3, 'calm': 4, 'happy': 5, 'sad': 6, 'angry': 7, 'fearful': 6, 'disgust': 1, 'surprised': 0}
Batch 7: {'neutral': 5, 'calm': 4, 'happy': 7, 'sad': 4, 'angry': 5, 'fearful': 4, 'disgust': 1, 'surprised': 2}
Batch 8: {'neutral': 5, 'calm': 6, 'happy': 7, 'sad': 2, 'angry': 0, 'fearful': 6, 'disgust': 3, 'surprised': 3}
Batch 9: {'neutral': 1, 'calm': 4, 'happy': 10, 'sad': 5, 'angry': 4, 'fearful': 1, 'disgust': 

In [36]:
ls = [[], [], []]
counter=0
for batch_pth in os.listdir(train_batches):
    batch = torch.load(os.path.join(train_batches, batch_pth))
    for i in range(len(batch[0])):
        ls[0].append(batch[0][i])
        ls[1].append(batch[1][i])
        ls[2].append(batch[2][i])
        counter+=1
        print(counter)

<ipython-input-36-50b0d77e1e4e>:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  batch = torch.load(os.path.join(train_batches, batch_pth))


Streaming output truncated to the last 5000 lines.
345
346
347
348
349
350
351
352
353
354
355
356
357
358
359
360
361
362
363
364
365
366
367
368
369
370
371
372
373
374
375
376
377
378
379
380
381
382
383
384
385
386
387
388
389
390
391
392
393
394
395
396
397
398
399
400
401
402
403
404
405
406
407
408
409
410
411
412
413
414
415
416
417
418
419
420
421
422
423
424
425
426
427
428
429
430
431
432
433
434
435
436
437
438
439
440
441
442
443
444
445
446
447
448
449
450
451
452
453
454
455
456
457
458
459
460
461
462
463
464
465
466
467
468
469
470
471
472
473
474
475
476
477
478
479
480
481
482
483
484
485
486
487
488
489
490
491
492
493
494
495
496
497
498
499
500
501
502
503
504
505
506
507
508
509
510
511
512
513
514
515
516
517
518
519
520
521
522
523
524
525
526
527
528
529
530
531
532
533
534
535
536
537
538
539
540
541
542
543
544
545
546
547
548
549
550
551
552
553
554
555
556
557
558
559
560
561
562
563
564
565
566
567
568
569
570
571
572
573
574
575
576
577
578
579
580
581
5

In [37]:
print(len(ls[0]))

5344


torch.Size([3, 35, 224, 224])


AttributeError: 'list' object has no attribute 'shape'

In [38]:
for i in range(32767):
    idx1 = random.randint(0, 5344)
    idx2 = random.randint(0, 5344)
    for i in range(len(ls)):
        ls[i][idx1], ls[i][idx2] = ls[i][idx2], ls[i][idx1]

IndexError: list index out of range

In [ ]:
ls_batch = []
curr_arr = [[], [], []]
for i in range(len(ls[0])):
    curr_arr[0].append(ls[0][i])
    curr_arr[1].append(ls[1][i])
    curr_arr[2].append(ls[2][i])
    if (len(curr_arr[0]) == 32):
        ls_batch.append(curr_arr)
        curr_arr = [[], [], []]

In [ ]:
global_cntr = {
    'neutral':0,
    'calm':0,
    'happy':0,
    'sad':0,
    'angry':0,
    'fearful':0,
    'disgust':0,
    'surprised':0
}
for i, batch in enumerate(ls_batch):
    cntr = {
        'neutral':0,
        'calm':0,
        'happy':0,
        'sad':0,
        'angry':0,
        'fearful':0,
        'disgust':0,
        'surprised':0
    }
    labels = batch[2]
    for label in labels:
        cntr[int_to_class[label]] += 1
        global_cntr[int_to_class[label]] += 1
    print(f'Batch {i+1}: {cntr}')

In [ ]:
cntr = {
    'neutral':0,
    'calm':0,
    'happy':0,
    'sad':0,
    'angry':0,
    'fearful':0,
    'disgust':0,
    'surprised':0
}
counter = 0
for label in ls[2]:
    cntr[int_to_class[label.item()]] += 1
    if (sum(cntr.values()) == 128):
        counter += 1
        print(f'batch {counter}: {cntr}')
        cntr = {
            'neutral':0,
            'calm':0,
            'happy':0,
            'sad':0,
            'angry':0,
            'fearful':0,
            'disgust':0,
            'surprised':0
        }

In [ ]:
for i in range(154, len(ls_batch)):
    video = torch.stack(ls_batch[i][0])
    #print(video.shape)
    audio = torch.stack(ls_batch[i][1])
    #print(audio.shape)
    label = torch.tensor(ls_batch[i][2])
    #print(label.shape)
    torch.save([video, audio, label], os.path.join(train_arrs, f'batch{i}.pt'))

In [ ]:
print(len(ls_batch))

In [ ]:
curr_arr = [[], [], []]
batch_cntr = 0
for i in range(len(ls[2])):
    for j in range(len(ls)):
        curr_arr[j].append(ls[j][i])
    if (len(curr_arr[0]) == 32):
        batch_cntr += 1
        curr_arr[0] = torch.stack(curr_arr[0])
        curr_arr[1] = torch.stack(curr_arr[1])
        curr_arr[2] = torch.tensor(curr_arr[2])
        torch.save(curr_arr, os.path.join(train_batches, 'batchnew' + str(batch_cntr) + '.pt'))
        curr_arr = [[], [], []]